# Official weight evaluation

This notebook runs the audited evaluation CLI with an offline, strictly loaded checkpoint. It defaults to a fixed eight-sample calibration; set `DFDHR_EVAL_MAX_SAMPLES=0` only for an intentional complete-dataset evaluation.

In [ ]:
import json
import os
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path


def required_path(name):
    value = os.environ.get(name)
    assert value, f"Missing required environment variable: {name}"
    return Path(value).expanduser().resolve()


repo_root = required_path("DFDHR_REPO_ROOT")
runtime_root = required_path("DFDHR_RUNTIME_ROOT")
data_root = required_path("DFDHR_DATA_ROOT")
checkpoint_path = required_path("DFDHR_OFFICIAL_CHECKPOINT")
assert repo_root.is_dir()
assert runtime_root.is_dir()
assert data_root.is_dir()
assert checkpoint_path.is_file()
assert repo_root not in runtime_root.parents and runtime_root != repo_root
assert data_root not in runtime_root.parents and runtime_root != data_root


In [ ]:
dataset_name = os.environ.get("DFDHR_EVAL_DATASET", "Celeb-DF-v2")
max_samples = int(os.environ.get("DFDHR_EVAL_MAX_SAMPLES", "8"))
assert max_samples >= 0
run_id = "notebook_official-eval_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
run_dir = runtime_root / "jupyter-validation" / run_id
run_dir.mkdir(parents=True, exist_ok=False)
report_path = run_dir / "metrics.json"

command = [
    sys.executable,
    "training/test.py",
    "--detector_path", "training/config/detector/dfd_hr.yaml",
    "--test_dataset", dataset_name,
    "--weights_path", str(checkpoint_path),
    "--architecture_only",
    "--test_batch_size", "1",
    "--workers", "0",
    "--output_path", str(report_path),
]
if max_samples:
    command.extend(["--max_samples_per_dataset", str(max_samples)])

environment = os.environ.copy()
environment["HF_HUB_OFFLINE"] = "1"
environment["TRANSFORMERS_OFFLINE"] = "1"
subprocess.run(command, cwd=repo_root, env=environment, check=True)


In [ ]:
with report_path.open(encoding="utf-8") as handle:
    report = json.load(handle)

assert report["checkpoint"]["sha256"]
assert report["dataset_json_sha256"][dataset_name]
assert report["evaluation"]["sample_counts"][dataset_name] > 0
print("Git commit:", report["code"]["git_commit"])
print("Dataset role:", dataset_name)
print("Sample count:", report["evaluation"]["sample_counts"][dataset_name])
print("Video count:", report["evaluation"]["video_counts"][dataset_name])
print("Metrics:", report["evaluation"]["metrics"][dataset_name])
print("Report directory role: DFDHR_RUNTIME_ROOT/jupyter-validation/<run-id>")
